In [ ]:
# 生成20行样本数据测试

import pandas as pd

# 读取原始 CSV 文件
# df = pd.read_csv('war_news_data.csv')

# # 随机抽取 20 行（如果数据不足 20 行，会报错，可加参数 ignore_index=True 或调整）
# sample_df = df.sample(n=100, random_state=42)  # random_state 用于结果可复现

# # 保存到新的 CSV 文件
# sample_df.to_csv('sample.csv', index=False)

# 读取原始 CSV 文件的前 2000 行
# df = pd.read_csv('war_news_data.csv', nrows=2000)

# 保存到新的 CSV 文件
# df.to_csv('first2000.csv', index=False)

In [ ]:

import os
import json
import time
import base64
import requests
import pandas as pd
import logging
from PIL import Image
from tqdm import tqdm

# ================= 配置区 =================
CSV_PATH = "sample.csv"
OUTPUT_PATH = "label_test_gemini-lite_4.json"
LOG_FILE = "dmx_labeling.log"

# DMXAPI 专用配置
MODEL = "gemini-2.5-flash-lite"
API_KEY = "sk-VEVYOwFHerqwvowhqqMJJuY5zJXWv5KafCUaJPMqv1EAUVVQ"
BASE_URL = "https://www.dmxapi.cn/v1beta"

# 代理配置 (按需开启)
# os.environ['http_proxy'] = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'


In [ ]:
SYSTEM_PROMPT = """
   **Role:** You are a senior scholar specializing in communication studies and visual psychology, with a focus on researching implicit biases and multimodal manipulation techniques in global news.

   **Task:** Please conduct an in-depth analysis of the provided [news headline/body] and [news image], and annotate them across multiple dimensions according to the **V-T-J-D Bias Evaluation System**."Regardless of the input language (Arabic, Chinese, English, etc.), please perform the semantic analysis based on the original cultural and linguistic context. The final analysis and JSON keys must remain in English for consistency."

   **Scoring Criteria for 0-3 Scale:**

   - **0 (None):** No evidence of the bias indicator. The presentation is neutral and purely factual.

   - **1 (Subtle):** Minor presence of the indicator. Might be accidental or a common journalistic practice, but slightly nudges the viewer's perception.

   - **2 (Obvious):** Clear intent to influence the viewer. Uses specific techniques (e.g., loaded words, dramatic angles) to frame the story.

   - **3 (Extreme):** Strong, systematic manipulation. Highly emotionalized or distorted presentation designed to provoke a specific reaction or dehumanize subjects.

------

   ### Bias Indicators Definition System:

   #### 1. Visual-Only Cues

   - **V1. Visual Salience Manipulation:** Whether certain details (e.g., painful tears, bloodstains) are deliberately highlighted through extreme close-ups, cropping, high saturation, or shallow depth of field while ignoring the overall context?

   - **V2. Subject-Centered Perspective & Social Distance**:

     > **Trigger Condition:** This indicator is marked as **True (Score 1-3)** ONLY when the primary subject is a **Human Being** (individual or group). If the image only contains objects, maps, or scenery without human presence, score **0**.

     **Scoring & Categorization (Must specify which case applies):**

     - **Low Angle (Worm’s Eye View):** Enhances the subject's status.
       - *Psychological Effect:* **Empowerment** (making a soldier look heroic), **Imposing** (making a leader look authoritative), or **Threatening** (making an aggressor look looming).
     - **High Angle (Bird’s Eye View):** Diminishes the subject's status.
       - *Psychological Effect:* **Victimization** (making a civilian look helpless), **Weakness** (showing subjects as small/insignificant), or **Pity** (inducing a sense of looking down upon suffering).
     - **Intimate Distance (Extreme Close-up):** Regardless of angle, if the camera is uncomfortably close to a human face.
       - *Psychological Effect:* **Forced Empathy** (invading the subject’s private space to highlight tears/pain, making the viewer feel a personal connection).
     - **Clinical Distance (Long Shot at Eye-level):** - *Psychological Effect:* **De-individualization** (treating humans as a faceless mass or statistics, common in "objective" but detached reporting).

   - **V3. Color/Lighting Rendering:** Whether unnatural color tones (e.g., cold blue filters implying evil; warm light suggesting justice) or strong contrasts between light and dark are used to create specific atmospheres?

   - **V4. Symbolic Visual Signs:** Are there elements such as national flags, police lines, religious symbols, ruins, etc., that carry strong cultural implications?

   #### 2. Text-Only Cues

   - **T1. Loaded Language Manipulation:** Does the text use emotionally charged vocabulary (e.g., "rioters" vs. "protesters", "invasion" vs. "action") to pre-establish positions?
   - **T2. Moral Judgment & Evaluation:** Does the text directly moralize about the subjects without factual support?

   #### 3. Joint/Interactive Biases (Image and Text)

   - **J1. Role Framing:** Do the image and text together construct clear dichotomies like "hero/villain" or "victim/aggressor"?

   - **J2. Selective Omission & Imbalance:** set J2 true if any of these apply：

     **Omission of relevant contextual facts**: key factual context is absent while relevant to interpretation (e.g., article shows protesters burning a building but omits that protest followed a lethal police action).

     **Single-sided imagery selection**: across the article, images show only one side’s violence/anger while text implies conflict between two sides.

     **Asymmetric humanization**: one side shown as individualized humans (faces, families) while the other side shown as faceless mobs/weapons or other subjects.

   - **J3. Stereotype Reinforcement:** Do the combinations of images and texts conform to stereotypes about specific ethnicities, classes, or cultures?


   #### 4. Multimodal Relationship Dimension

   - D1. Relationship Qualification:

     Goal: Determine which modality primarily controls interpretation.
     Choose ONE category by strictly following the order below (do NOT skip).

     - Tension / Mismatch
       Use if image and text guide interpretation in conflicting directions (e.g., calm text + chaotic image, irony, contradiction).
       Test: Would viewers feel cognitive dissonance?

     - Visual Amplification
       Use if text is factual/neutral AND image is emotionally or morally intense, driving audience reaction.
       Test: Remove image → emotional impact largely disappears.

     - Anchoring
       Use if image alone is ambiguous/neutral AND text assigns strong qualitative labels (e.g., “invaders”, “heroes”).
       Test: Remove text → image meaning becomes unclear.

     - Reinforcing (NOT default)
       Use ONLY if image alone AND text alone independently express the SAME evaluative stance, and neither dominates.
       Test: Remove either modality → remaining one still conveys same judgment.

     Neutral
     Use ONLY if both image and text are descriptive, non-evaluative, and emotionally neutral (rare in conflict news).

     Output ONE label only.

     **To reduce "Reinforcing" overclassification:** apply Reinforcing only if BOTH:

     1. Text uses evaluative words or clear framing *and* image visually supports the same evaluative framing (e.g., text calls group "aggressors" and image is low-angle of armed people), 

     2. There are no more specific categories above (Anchoring, Amplification, Tension).

        D2. Coherence Score:

     **Coherence_Score mapping (0–5):**
        Calculate based on:

        - Semantic Alignment (0–2):
          0: different events / meanings
             1: same event, different emphasis
             2: same event, same interpretation

        - Emotional Alignment (0–2):
          0: conflicting emotions
             1: weak or one-sided emotion
             2: shared emotional tone

        - Direction Consistency (0–1)
          0: pull interpretation in different directions

             1: guide interpretation in same direction
          ---

          Coherence_Score = Semantic Alignment (0–2) + Emotional Alignment (0–2) + Direction Consistency (0–1)



## Output Requirements

### 1. Analysis

Briefly outline:

- Visual composition, perspective, color, and human status
- Key textual cues and tones
- How image and text interact semantically

### 2. JSON Format (STRICT)

{
  "analysis_chain": {
    "visual_observation": "string",
    "textual_observation": "string",
    "joint_mechanism": "string"
  },
  "cues": {
    "visual_only": {
      "V1_Salience": {"present": boolean, "score": 0-3, "reason": "string"},
      "V2_Perspective": {
        "present": boolean,
        "score": 0-3,
        "reason": "string",
        "analysis": {
          "subject_type": "Soldier / Civilian / Leader / Mass Crowd / Other Human / No Human",
          "camera_angle": "Low / High / Eye-level / Extreme Close-up / None"
        }
      },
      "V3_Color_Lighting": {"present": boolean, "score": 0-3, "reason": "string"},
      "V4_Symbolism": {"present": boolean, "score": 0-3, "reason": "string"}
    },
    "text_only": {
      "T1_Loaded_Language": {"present": boolean, "score": 0-3, "reason": "string"},
      "T2_Moral_Judgment": {"present": boolean, "score": 0-3, "reason": "string"}
    },
    "joint_multimodal": {
      "J1_Role_Framing": {"present": boolean, "reason": "string"},
      "J2_Selective_Imbalance": {"present": boolean, "reason": "string"},
      "J3_Stereotyping": {"present": boolean, "reason": "string"}
    },
    "consistency_dimension": {
      "D1_Relationship_Type": "Reinforcing / Tension / Neutral / Anchoring / Amplification",
      "Coherence_Score": 0-5,
      "reason": "string"
    }
  },
  "overall_bias_intensity": 0-5
}

"""

In [ ]:
# ================= 工具函数 =================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler(LOG_FILE, encoding='utf-8')]
)
logger = logging.getLogger(__name__)

def encode_image(image_path):
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def load_progress():
    if os.path.exists(OUTPUT_PATH):
        try:
            with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data, {str(item.get('news_id')) for item in data}
        except: pass
    return [], set()

In [ ]:
# ================= 核心请求：针对 DMXAPI 的流式死磕函数 =================
def call_dmx_api_persistent(image_path, prompt_text, pbar, news_id):
    """
    针对 DMXAPI SSE 接口的无限重试函数
    """
    url = f"{BASE_URL}/models/{MODEL}:streamGenerateContent?key={API_KEY}&alt=sse"
    image_base64 = encode_image(image_path)
    
    headers = {"Content-Type": "application/json"}
    payload = {
        "contents": [{
            "parts": [
                {"inline_data": {"mime_type": "image/jpeg", "data": image_base64}},
                {"text": f"{SYSTEM_PROMPT}\n\n{prompt_text}"}
            ]
        }]
    }

    wait_time = 10
    retry_count = 0

    while True:
        try:
            # 使用 requests 发送 POST，开启流式获取
            # 注意：此处 stream=True 配合 iter_lines 解析 SSE
            response = requests.post(url, headers=headers, json=payload, timeout=60, stream=True)
            
            # 1. 检查状态码
            if response.status_code == 429:
                raise requests.exceptions.HTTPError("429 Too Many Requests")
            response.raise_for_status()

            full_text = ""
            # 2. 解析 SSE 响应
            for line in response.iter_lines():
                if not line: continue
                line_str = line.decode('utf-8')
                # 跳过结束标记
                if line_str.strip() == 'data: [DONE]': 
                    continue
                if line_str.startswith('data: '):
                    json_str = line_str[6:]
                    try:
                        data = json.loads(json_str)
                        # 提取文本块
                        if 'candidates' in data:
                            for cand in data['candidates']:
                                for part in cand.get('content', {}).get('parts', []):
                                    if 'text' in part:
                                        full_text += part['text']
                    except Exception:
                        # 有时 SSE 中的片段本身不是完整 JSON，继续拼接
                        try:
                            full_text += json_str
                        except: 
                            continue

            # 3. 验证 JSON 完整性并返回（更鲁棒的提取）
            # 去掉常见包裹标记
            final_json_str = full_text.strip()
            final_json_str = final_json_str.replace('```json', '').replace('```', '').strip()

            # 尝试从第一个 '{' 到最后一个 '}' 提取合法 JSON 子串
            first_brace = final_json_str.find('{')
            last_brace = final_json_str.rfind('}')
            if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                candidate = final_json_str[first_brace:last_brace+1]
            else:
                candidate = final_json_str  # 作为兜底

            try:
                return json.loads(candidate)
            except Exception as e:
                # 保存原始响应用于调试
                dbg_path = f"debug_resp_{news_id}.txt"
                with open(dbg_path, 'w', encoding='utf-8') as dbgf:
                    dbgf.write("=== full_text ===\n")
                    dbgf.write(full_text + "\n\n=== final_json_str ===\n")
                    dbgf.write(final_json_str + "\n\n=== candidate ===\n")
                    dbgf.write(candidate)
                logger.error(f"❌ ID:{news_id} JSON 解析失败，已保存调试文件: {dbg_path} | 错误: {e}")
                raise e
            
            
        except (requests.exceptions.HTTPError, requests.exceptions.ConnectionError) as e:
            # 针对 429 或 网络闪断 的死磕逻辑
            retry_count += 1
            status_desc = "429配额耗尽" if "429" in str(e) else "网络异常"
            pbar.set_description(f"⏳ [{status_desc}] ID:{news_id} | 重试 {retry_count} | 待机 {wait_time}s")
            logger.warning(f"ID:{news_id} {status_desc}, sleep {wait_time}s")
            
            time.sleep(wait_time)
            wait_time = min(wait_time + 10, 60) # 逐渐增加等待时间，封顶60秒
            continue

        except Exception as e:
            # 针对其他错误 (JSON 解析失败或 400 错误)
            logger.error(f"❌ ID:{news_id} 发生不可恢复错误: {e}")
            raise e

In [ ]:
# ================= 主流程 =================
def start_labeling():
    df = pd.read_csv(CSV_PATH)
    df['news_id'] = df['news_id'].astype(str)
    
    results, processed_ids = load_progress()
    to_process = df[~df['news_id'].isin(processed_ids)]
    
    print(f"🚀 DMXAPI 任务启动 | 待处理: {len(to_process)}")
    
    pbar = tqdm(to_process.iterrows(), total=len(to_process), ncols=140)
    success_count = 0

    for _, row in pbar:
        aid = row['news_id']
        title = str(row['title'])
        image_path = row['image_path']
        text_path = row['text_path']
        pbar.set_description(f"⚡ 正在分析 [{aid}]")

        if not os.path.exists(row['image_path']):
            pbar.write(f"⚠️  [跳过] ID: {aid} | 图片缺失")
            continue

        try:
            # 读取文本描述
            try:
                with open(row['text_path'], 'r', encoding='utf-8') as f: text_content = f.read()
            except: text_content = str(row['description'])

            # 执行死磕请求
            res_data = call_dmx_api_persistent(row['image_path'], f"Title: {row['title']}\nContent: {text_content}", pbar, aid)
            
            # 处理结果
            res_data['news_id'] = aid
            res_data['title'] = title
            res_data['image_path'] = image_path
            res_data['text_path'] = text_path

            results.append(res_data)
            success_count += 1
            
            pbar.write(f"✅ [OK] ID: {aid} | 标题: {str(row['title'])[:15]}...")

            # 定期存盘
            if success_count % 5 == 0:
                with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
                    json.dump(results, f, ensure_ascii=False, indent=2)
            
            # DMXAPI 通常也有请求间隔建议，手动休眠 1-2 秒更安全
            time.sleep(1)

        except Exception as e:
            pbar.write(f"❌ [跳过] ID: {aid} | 错误: {str(e)[:50]}")
            continue

    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print("\n🎉 所有数据标注已完成！")

In [ ]:
# 执行任务
if __name__ == "__main__":
    start_labeling()

In [ ]:
import json
